# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuratulainAzhar22/flyrank-ml-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Initial Setup

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("My_Read_Token")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection configured.")

Hugging Face connection configured.


In [8]:
march_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT *
FROM read_parquet('{march_path}')
""")

print("March 2026 view created.")

March 2026 view created.


In [2]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


I will use two signals for my baseline rule:

1. **Average position / CTR relationship — CONFIRMED**
   The observed CTR decreases as position gets worse. CTR is highest for positions 1–3 (0.003814), falls to 0.003041 for positions 4–10, remains around 0.003056 for positions 11–20, and falls to 0.001262 for positions 21+. This supports using position and CTR as signals for identifying pages that may have an opportunity to improve search performance.

2. **Impression volume — CONFIRMED**
   Impression volume provides a useful opportunity signal. The data contains substantially more total impressions in the higher-volume buckets: 535,373,900 for <10 impressions, 548,147,670 for 10–99, 1,616,094,300 for 100–999, and 588,796,530 for 1000+. Higher-volume rows provide more potential search exposure, so volume is useful for prioritizing actions.

### Baseline rule

I will prioritize rows that have meaningful search visibility but relatively weak CTR for their position. The score will combine:

- search impression volume,
- position,
- and CTR weakness relative to the observed position buckets.

The queue will rank rows by this score.

### Reason codes

- `CTR_POSITION_OPPORTUNITY` — the row has search visibility and its CTR is weak relative to its position bucket.
- `HIGH_VOLUME_OPPORTUNITY` — the row has high search impression volume, making the potential opportunity larger.
- `LOW_PRIORITY` — the row does not show a strong enough combination of visibility and CTR opportunity.

The rule is a decision-support baseline, not a prediction of future performance.

Signal 1 — CTR vs Position

Your actual fields are:

gsc_clicks
gsc_impressions
gsc_avg_position

CTR is calculated as:

clicks / impressions

In [9]:
signal_1 = con.sql("""
SELECT
    CASE
        WHEN gsc_avg_position < 4 THEN '1-3'
        WHEN gsc_avg_position < 11 THEN '4-10'
        WHEN gsc_avg_position < 21 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,

    COUNT(*) AS n,

    SUM(gsc_clicks) AS clicks,
    SUM(gsc_impressions) AS impressions,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr

FROM march
WHERE gsc_data_available IS TRUE
GROUP BY 1
ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        ELSE 4
    END
""").df()

signal_1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,clicks,impressions,ctr
0,1-3,957386,340806.0,89360890.0,0.003814
1,4-10,1293073,324275.0,106618470.0,0.003041
2,11-20,483543,84986.0,27807032.0,0.003056
3,21+,877059,71765.0,56871197.0,0.001262


### Signal 1 verdict: CONFIRMED

The measured CTR changes across position buckets, so CTR should be interpreted relative to position rather than treated as a single global threshold.

This supports using a position-adjusted CTR signal in the baseline.

Signal 2 — Search volume

Your volume signal is:

gsc_impressions

We'll divide it into buckets.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal_2 = con.sql("""
SELECT
    CASE
        WHEN gsc_impressions < 10 THEN '<10'
        WHEN gsc_impressions < 100 THEN '10-99'
        WHEN gsc_impressions < 1000 THEN '100-999'
        ELSE '1000+'
    END AS impressions_bucket,

    COUNT(*) AS n,

    AVG(gsc_impressions) AS avg_impressions,
    SUM(gsc_impressions) AS total_impressions

FROM march
WHERE gsc_data_available IS TRUE
GROUP BY 1
ORDER BY
    CASE impressions_bucket
        WHEN '<10' THEN 1
        WHEN '10-99' THEN 2
        WHEN '100-999' THEN 3
        ELSE 4
    END
""").df()

signal_2

,impressions_bucket,n,avg_impressions,total_impressions
0,<10,1463532,3.658095,5353739.0
1,10-99,1508921,36.327128,54814767.0
2,100-999,606189,266.599080,161609430.0
3,1000+,32419,1816.208180,58879653.0


### Signal 2 verdict: CONFIRMED

Search volume varies substantially across impression buckets. This means impression volume can help distinguish a weak CTR signal with meaningful search exposure from a weak CTR signal with very little evidence.

I will therefore use impressions as a supporting priority signal rather than treating every low-CTR row equally.

Impression volume is useful as a prioritization/opportunity signal.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Now we make the actual baseline.

The idea is:

Low CTR for the page's position + enough impressions = higher priority.

We need a position-adjusted CTR benchmark.

We'll calculate the average CTR for each position bucket.

In [11]:
position_benchmarks = con.sql("""
SELECT
    CASE
        WHEN gsc_avg_position < 4 THEN '1-3'
        WHEN gsc_avg_position < 11 THEN '4-10'
        WHEN gsc_avg_position < 21 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,

    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS benchmark_ctr

FROM march
WHERE
    gsc_data_available IS TRUE
    AND gsc_impressions > 0
    AND gsc_avg_position IS NOT NULL

GROUP BY 1
""").df()

position_benchmarks

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,benchmark_ctr
0,1-3,0.003814
1,11-20,0.003056
2,4-10,0.003041
3,21+,0.001262


Now create the score

In [12]:
queue = con.sql("""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_avg_position < 4 THEN '1-3'
            WHEN gsc_avg_position < 11 THEN '4-10'
            WHEN gsc_avg_position < 21 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,

        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS ctr

    FROM march

    WHERE
        gsc_data_available IS TRUE
        AND gsc_impressions > 0
        AND gsc_avg_position IS NOT NULL
),

benchmarks AS (
    SELECT
        CASE
            WHEN gsc_avg_position < 4 THEN '1-3'
            WHEN gsc_avg_position < 11 THEN '4-10'
            WHEN gsc_avg_position < 21 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,

        SUM(gsc_clicks) * 1.0 /
        NULLIF(SUM(gsc_impressions), 0) AS benchmark_ctr

    FROM march

    WHERE
        gsc_data_available IS TRUE
        AND gsc_impressions > 0
        AND gsc_avg_position IS NOT NULL

    GROUP BY 1
),

scored AS (
    SELECT
        b.*,
        p.benchmark_ctr,

        p.benchmark_ctr - b.ctr AS ctr_gap,

        CASE
            WHEN b.gsc_impressions >= 100
                 AND b.ctr < p.benchmark_ctr
            THEN 2

            WHEN b.gsc_impressions < 100
                 AND b.ctr < p.benchmark_ctr
            THEN 1

            ELSE 0
        END AS score

    FROM base b

    LEFT JOIN benchmarks p
        ON b.position_bucket = p.position_bucket
)

SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ctr,
    benchmark_ctr,
    ctr_gap,
    score,

    CASE
        WHEN score = 2
            THEN 'HIGH_VOLUME_CTR_GAP'

        WHEN score = 1
            THEN 'CTR_POSITION_GAP'

        ELSE 'NO_STRONG_SIGNAL'
    END AS reason_code,

    CASE
        WHEN score = 2
            THEN 'PRIORITIZE_REVIEW'

        WHEN score = 1
            THEN 'REVIEW_CTR'

        ELSE 'MONITOR'
    END AS action

FROM scored

ORDER BY score DESC, ctr_gap DESC NULLS LAST
""").df()

queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,benchmark_ctr,ctr_gap,score,reason_code,action
0,client_73cda7b4e4f265ea,content_712c365258cee05c,2026-03-01,223,0,3.892377,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
1,client_73cda7b4e4f265ea,content_347a278bb1a646f5,2026-03-01,134,0,2.082090,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
2,client_73cda7b4e4f265ea,content_4cd532e5e9edc02b,2026-03-01,205,0,3.634146,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
3,client_73cda7b4e4f265ea,content_c6edc8d0083a4f2e,2026-03-01,203,0,3.733990,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
4,client_73cda7b4e4f265ea,content_a15138d47e9949ae,2026-03-01,531,0,0.802260,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
5,client_73cda7b4e4f265ea,content_d8a7baed69149f40,2026-03-01,356,0,1.924157,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
6,client_73cda7b4e4f265ea,content_dc50704a2abee2df,2026-03-01,215,0,2.883721,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
7,client_73cda7b4e4f265ea,content_3ea1da426a358e8f,2026-03-01,361,0,0.429363,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
8,client_73cda7b4e4f265ea,content_db9fee50d56d8f1e,2026-03-01,103,0,1.155340,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
9,client_73cda7b4e4f265ea,content_af63c445e6d77e10,2026-03-01,134,0,0.634328,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW


**rule meaning:**

Score = 2
CTR below expected CTR for its position
+
100+ impressions

Action:

PRIORITIZE_REVIEW

Reason:

HIGH_VOLUME_CTR_GAP
Score = 1
CTR below expected CTR for its position
+
less than 100 impressions
Action:

REVIEW_CTR

Reason:

CTR_POSITION_GAP
Score = 0
No strong CTR-vs-position signal

Action:

MONITOR

Reason:

NO_STRONG_SIGNAL

This is a transparent rule, not a machine-learning model.

**Write the CSV**

In [13]:
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Saved ranked queue to: {output_path}")
print(f"Rows written: {len(queue):,}")

Saved ranked queue to: work/outputs/baseline_action_score.csv
Rows written: 3,611,061


In [14]:
queue.head(20)

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,benchmark_ctr,ctr_gap,score,reason_code,action
0,client_73cda7b4e4f265ea,content_712c365258cee05c,2026-03-01,223,0,3.892377,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
1,client_73cda7b4e4f265ea,content_347a278bb1a646f5,2026-03-01,134,0,2.082090,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
2,client_73cda7b4e4f265ea,content_4cd532e5e9edc02b,2026-03-01,205,0,3.634146,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
3,client_73cda7b4e4f265ea,content_c6edc8d0083a4f2e,2026-03-01,203,0,3.733990,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
4,client_73cda7b4e4f265ea,content_a15138d47e9949ae,2026-03-01,531,0,0.802260,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
5,client_73cda7b4e4f265ea,content_d8a7baed69149f40,2026-03-01,356,0,1.924157,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
6,client_73cda7b4e4f265ea,content_dc50704a2abee2df,2026-03-01,215,0,2.883721,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
7,client_73cda7b4e4f265ea,content_3ea1da426a358e8f,2026-03-01,361,0,0.429363,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
8,client_73cda7b4e4f265ea,content_db9fee50d56d8f1e,2026-03-01,103,0,1.155340,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
9,client_73cda7b4e4f265ea,content_af63c445e6d77e10,2026-03-01,134,0,0.634328,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


I reviewed the first 20 rows of the ranked queue manually.

For each row I considered:
- the recommended action,
- the reason code,
- how strong the evidence appears,
- and what could make the recommendation wrong.

The ranking is decision-support only. A high score does not prove that a page needs a specific intervention.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_avg_position",
        "ctr",
        "benchmark_ctr",
        "ctr_gap",
        "score",
        "reason_code",
        "action"
    ]
]

,content_hash_id,gsc_impressions,gsc_avg_position,ctr,benchmark_ctr,ctr_gap,score,reason_code,action
0,content_712c365258cee05c,223,3.892377,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
1,content_347a278bb1a646f5,134,2.082090,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
2,content_4cd532e5e9edc02b,205,3.634146,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
3,content_c6edc8d0083a4f2e,203,3.733990,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
4,content_a15138d47e9949ae,531,0.802260,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
5,content_d8a7baed69149f40,356,1.924157,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
6,content_dc50704a2abee2df,215,2.883721,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
7,content_3ea1da426a358e8f,361,0.429363,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
8,content_db9fee50d56d8f1e,103,1.155340,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW
9,content_af63c445e6d77e10,134,0.634328,0.0,0.003814,0.003814,2,HIGH_VOLUME_CTR_GAP,PRIORITIZE_REVIEW


| Rank | Action            | Reason                                                   | Confidence | What would make it wrong?                                                                                                   |
| ---: | ----------------- | -------------------------------------------------------- | ---------- | --------------------------------------------------------------------------------------------------------------------------- |
|    1 | PRIORITIZE_REVIEW | 223 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The recorded 0 CTR could reflect missing, rounded, or tracking data rather than a real lack of clicks.                      |
|    2 | PRIORITIZE_REVIEW | 134 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The zero CTR may be a measurement/data-quality issue rather than a genuine opportunity.                                     |
|    3 | PRIORITIZE_REVIEW | 205 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The CTR value could be incomplete or affected by tracking/reporting limitations.                                            |
|    4 | PRIORITIZE_REVIEW | 203 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The apparent CTR gap may not represent a real improvement opportunity if the CTR is unreliable.                             |
|    5 | PRIORITIZE_REVIEW | 531 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The high impression count does not guarantee that the zero CTR is actionable; measurement quality could be the issue.       |
|    6 | PRIORITIZE_REVIEW | 356 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The zero CTR could be caused by incomplete or delayed search data.                                                          |
|    7 | PRIORITIZE_REVIEW | 215 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The observed zero CTR may be a data artifact rather than a genuine CTR problem.                                             |
|    8 | PRIORITIZE_REVIEW | 361 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The recommendation could be wrong if the recorded CTR does not accurately represent actual clicks.                          |
|    9 | PRIORITIZE_REVIEW | 103 impressions with 0 CTR and a 0.003814 benchmark gap. | Low        | The relatively small impression count makes the signal less stable and could make the opportunity look stronger than it is. |
|   10 | PRIORITIZE_REVIEW | 134 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The zero CTR may reflect measurement limitations rather than true search underperformance.                                  |
|   11 | PRIORITIZE_REVIEW | 142 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The recorded CTR could be incomplete or rounded to zero.                                                                    |
|   12 | PRIORITIZE_REVIEW | 147 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The apparent gap may disappear if the underlying click data is corrected or updated.                                        |
|   13 | PRIORITIZE_REVIEW | 122 impressions with 0 CTR and a 0.003814 benchmark gap. | Low        | The relatively low impression volume makes this a weaker decision-support signal.                                           |
|   14 | PRIORITIZE_REVIEW | 704 impressions with 0 CTR and a 0.003814 benchmark gap. | High       | The zero CTR could still be caused by a tracking or data-quality problem despite the strong impression volume.              |
|   15 | PRIORITIZE_REVIEW | 319 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The recommendation would be weaker if the zero CTR is caused by incomplete reporting.                                       |
|   16 | PRIORITIZE_REVIEW | 202 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The recorded zero CTR may not represent the true click rate.                                                                |
|   17 | PRIORITIZE_REVIEW | 332 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The rule could be wrong if the CTR value is stale, incomplete, or affected by measurement issues.                           |
|   18 | PRIORITIZE_REVIEW | 177 impressions with 0 CTR and a 0.003814 benchmark gap. | Medium     | The apparent opportunity may be caused by a reporting artifact rather than genuine underperformance.                        |
|   19 | PRIORITIZE_REVIEW | 127 impressions with 0 CTR and a 0.003814 benchmark gap. | Low        | The relatively low impression volume makes the zero-CTR signal less reliable for prioritization.                            |
|   20 | PRIORITIZE_REVIEW | 525 impressions with 0 CTR and a 0.003814 benchmark gap. | High       | The recommendation could still be wrong if the zero CTR reflects tracking or reporting problems.                            |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks are the rows with relatively low impression counts, especially ranks 9, 13, 19, 2, 10, 11, and 12. Their zero CTR creates the same benchmark gap as the higher-volume rows, but the smaller amount of exposure makes the decision less stable.

The baseline uses current-period search signals only: impressions, average position, CTR, and the position-based benchmark CTR. I did not use the future-window label or any label-derived column in the ranking rule. Therefore, the baseline is intended as a decision-support ranking rather than a future-performance prediction.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
used_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("Columns used by baseline:")
for col in used_columns:
    print("-", col)

Columns used by baseline:
- gsc_impressions
- gsc_clicks
- gsc_avg_position


In [17]:
forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win"
]

available_forbidden = [
    c for c in forbidden_columns
    if c in queue.columns
]

print("Forbidden/product-label columns present in queue:")
print(available_forbidden)

assert len(available_forbidden) == 0
print("Leakage check passed.")

Forbidden/product-label columns present in queue:
[]
Leakage check passed.


**Limitation**

A limitation of this baseline is that the performance table does not contain a direct content-age or publication-date field, so I could not test a true staleness signal. I therefore used CTR-vs-position and search volume, which are directly observable in the available March data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.